In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import chart_studio.plotly as ply
import os
from scipy import interpolate as interp
from scipy.signal import spectrogram, firwin, filtfilt, hilbert
from utilities.signal_analysis_utils import get_harmonic_envelope, get_total_envelope
import utm

In [ ]:
from types import SimpleNamespace

# Scans to put on the same map, in the order they were recorded (mag file, gps file)
scan_files = [
    ('data/mag1.csv', 'data/gps1.csv'),
    ('data/mag.csv', 'data/gps.csv'),
]

# Magnetic secondary coil data acquisition rate and primary coil driving frequency
sample_rate = int(2e3) # in [Hz]
driving_frequency = 100 # in [Hz]

# Set to False to keep the data recorded during turns on the maps
remove_turns = True
# A GPS point counts as part of a turn when the heading changes faster than this, in [deg/s]
turn_rate_threshold = 10

Processing of Each Scan: RMS Voltages, Harmonic Envelopes, Turn Removal, and Interpolation to GPS Coordinates

Each scan is processed on its own (so filters never run across the gap between scans) and the results are combined afterwards.

In [ ]:
def process_scan(mag_path, gps_path):
    mag_data = pd.read_csv(mag_path)
    mag_times = mag_data['Times (s)'].to_numpy() # in [s]
    mag_ch1_voltages = mag_data['Ch 1 (V)'].to_numpy()
    mag_ch2_voltages = mag_data['Ch 2 (V)'].to_numpy()
    mag_primary_voltages = mag_data['Primary (V)'].to_numpy()
    # AC voltages computed by subtracting means from raw voltages
    mag_ch1_ac_voltages = mag_ch1_voltages-np.mean(mag_ch1_voltages)
    mag_ch2_ac_voltages = mag_ch2_voltages-np.mean(mag_ch2_voltages)
    mag_primary_ac_voltages = mag_primary_voltages-np.mean(mag_primary_voltages)

    gps_data = pd.read_csv(gps_path)
    gps_times = gps_data['Times (s)'].to_numpy()
    gps_lat = gps_data['Latitude'].to_numpy()
    gps_lon = gps_data['Longitude'].to_numpy()
    gps_headings = gps_data['Heading'].to_numpy()

    # Computes RMS voltages
    num_rms_bins = len(mag_ch1_ac_voltages) // sample_rate
    mag_rms_times = mag_times[np.arange(num_rms_bins) * sample_rate]
    mag_ch1_rms_voltages = np.array([
        np.sqrt(np.mean(np.square(mag_ch1_ac_voltages[i * sample_rate:(i + 1) * sample_rate])))
        for i in range(num_rms_bins)
    ])
    mag_ch2_rms_voltages = np.array([
        np.sqrt(np.mean(np.square(mag_ch2_ac_voltages[i * sample_rate:(i + 1) * sample_rate])))
        for i in range(num_rms_bins)
    ])

    # Flags 1 s bins where the transmitter is running (it shuts off a few seconds before each recording ends)
    primary_bin_std = np.array([
        np.std(mag_primary_ac_voltages[i * sample_rate:(i + 1) * sample_rate])
        for i in range(num_rms_bins)
    ])
    transmitter_on_bins = primary_bin_std > 0.9 * np.median(primary_bin_std)
    transmitter_on_samples = np.zeros(len(mag_times), dtype=bool)
    transmitter_on_samples[:num_rms_bins * sample_rate] = np.repeat(transmitter_on_bins, sample_rate)

    # Means are taken over transmitter-on data only, so the dead seconds at the end don't shift the baseline
    mag_ch1_rms_voltages -= np.mean(mag_ch1_rms_voltages[transmitter_on_bins])
    mag_ch2_rms_voltages -= np.mean(mag_ch2_rms_voltages[transmitter_on_bins])

    # Computes harmonic envelopes
    mag_ch1_fundamental_envelope = get_harmonic_envelope(mag_ch1_ac_voltages, sample_rate, mag_primary_ac_voltages, driving_frequency, 1)[2]
    mag_ch1_fundamental_envelope = mag_ch1_fundamental_envelope - np.mean(mag_ch1_fundamental_envelope[transmitter_on_samples])
    mag_ch2_fundamental_envelope = get_harmonic_envelope(mag_ch2_ac_voltages, sample_rate, mag_primary_ac_voltages, driving_frequency, 1)[2]
    mag_ch2_fundamental_envelope = mag_ch2_fundamental_envelope - np.mean(mag_ch2_fundamental_envelope[transmitter_on_samples])

    # Flags GPS points in turns: heading changing faster than turn_rate_threshold, measured over +-2 s
    # (single-second GPS heading jitter on straight lines can exceed 5 deg, real turns are 25-35 deg/s).
    # Differences are wrapped so 359 -> 1 deg counts as a 2 deg change, not 358 deg.
    point = np.arange(len(gps_headings))
    ahead = np.minimum(point + 2, len(point) - 1)
    behind = np.maximum(point - 2, 0)
    heading_change = (gps_headings[ahead] - gps_headings[behind] + 180) % 360 - 180
    turn_rate = heading_change / np.maximum(ahead - behind, 1)
    in_turn = np.abs(turn_rate) > turn_rate_threshold
    # Adds a 1-point margin before and after each turn
    in_turn = in_turn | np.r_[False, in_turn[:-1]] | np.r_[in_turn[1:], False]

    # Flags GPS points whose interpolated values only use transmitter-on bins (the bin it falls in and its neighbours)
    gps_bins = np.floor(gps_times - mag_times[0]).astype(int)
    transmitter_on = np.array([
        1 <= b < num_rms_bins - 1 and transmitter_on_bins[b - 1:b + 2].all() for b in gps_bins
    ])

    # Interpolates RMS magnetic data to GPS coordinates
    mag_ch1_voltages_interp = interp.interp1d(mag_rms_times, mag_ch1_rms_voltages, kind='cubic', fill_value='extrapolate')
    mag_ch2_voltages_interp = interp.interp1d(mag_rms_times, mag_ch2_rms_voltages, kind='cubic', fill_value='extrapolate')
    mag_ch1_voltages_at_gps_coords = mag_ch1_voltages_interp(gps_times)
    mag_ch2_voltages_at_gps_coords = mag_ch2_voltages_interp(gps_times)

    # Interpolates fundamental frequency magnetic data to GPS coordinates
    mag_ch1_fundamental_envelope_interp = interp.interp1d(mag_times, mag_ch1_fundamental_envelope, kind='cubic', fill_value='extrapolate')
    mag_ch1_fundamental_envelope_at_gps_coords = mag_ch1_fundamental_envelope_interp(gps_times)
    mag_ch2_fundamental_envelope_interp = interp.interp1d(mag_times, mag_ch2_fundamental_envelope, kind='cubic', fill_value='extrapolate')
    mag_ch2_fundamental_envelope_at_gps_coords = mag_ch2_fundamental_envelope_interp(gps_times)

    name = os.path.basename(mag_path)
    # Returns every variable above, e.g. scans[0].mag_times
    return SimpleNamespace(**locals())

scans = [process_scan(mag_path, gps_path) for mag_path, gps_path in scan_files]

In [ ]:
for s in scans:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x= s.mag_rms_times-s.mag_times[0],
        y= s.mag_ch1_rms_voltages,
        mode='lines',
        name='Ch 1 RMS Voltages',
        line=dict(color='rebeccapurple', width=2)
    ))
    fig.add_trace(go.Scatter(
        x= s.mag_times-s.mag_times[0],
        y= s.mag_ch1_fundamental_envelope,
        mode='lines',
        name='Ch 1 Fundamental Envelope',
        line=dict(color='royalblue', width=2)
    ))
    fig.update_layout(
        title=f'Ch 1 RMS Voltage and Fundamental Envelope vs. Time ({s.name})',
        xaxis_title='Time (s)',
        yaxis_title='Voltage (V)',
        template='plotly_dark',
        hovermode='x unified',
    )
    fig.show()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x= s.gps_times[1:]-s.gps_times[0],
        y= s.gps_headings[1:]-s.gps_headings[:-1],
        mode='lines',
        name='GPS Heading Change',
        line=dict(color='rebeccapurple', width=2),
        showlegend=True
    ))
    fig.update_layout(
        title=f'Heading Change vs. Time ({s.name})',
        xaxis_title='Time (s)',
        yaxis_title='Heading Change (degrees)',
        template='plotly_dark',
        hovermode='x unified',
    )
    fig.show()

After Removing Turns and Transmitter-Off Points (values at GPS points; removed points are gaps)

In [ ]:
for s in scans:
    removed = s.in_turn | ~s.transmitter_on
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x= s.gps_times-s.mag_times[0],
        y= np.where(removed, np.nan, s.mag_ch1_voltages_at_gps_coords),
        mode='lines+markers',
        name='Ch 1 RMS Voltages',
        line=dict(color='rebeccapurple', width=2)
    ))
    fig.add_trace(go.Scatter(
        x= s.gps_times-s.mag_times[0],
        y= np.where(removed, np.nan, s.mag_ch1_fundamental_envelope_at_gps_coords),
        mode='lines+markers',
        name='Ch 1 Fundamental Envelope',
        line=dict(color='royalblue', width=2)
    ))
    fig.update_layout(
        title=f'Ch 1 RMS Voltage and Fundamental Envelope vs. Time, Turns Removed ({s.name})',
        xaxis_title='Time (s)',
        yaxis_title='Voltage (V)',
        template='plotly_dark',
        hovermode='x unified',
    )
    fig.show()

Combining the Scans onto One Set of GPS Points

In [ ]:
def combine(attribute):
    return np.concatenate([getattr(s, attribute) for s in scans])

gps_times = combine('gps_times')
gps_lat = combine('gps_lat')
gps_lon = combine('gps_lon')
mag_ch1_voltages_at_gps_coords = combine('mag_ch1_voltages_at_gps_coords')
mag_ch2_voltages_at_gps_coords = combine('mag_ch2_voltages_at_gps_coords')
mag_ch1_fundamental_envelope_at_gps_coords = combine('mag_ch1_fundamental_envelope_at_gps_coords')
mag_ch2_fundamental_envelope_at_gps_coords = combine('mag_ch2_fundamental_envelope_at_gps_coords')
# Which scan (0, 1, ...) each GPS point came from
scan_index = np.concatenate([np.full(len(s.gps_times), k) for k, s in enumerate(scans)])
# Position of each GPS point within its own scan, counted from the start and from the end
position_in_scan = np.concatenate([np.arange(len(s.gps_times)) for s in scans])
position_from_scan_end = np.concatenate([np.arange(len(s.gps_times))[::-1] for s in scans])
in_turn = combine('in_turn')
transmitter_on = combine('transmitter_on')
# GPS points that go on the maps and into the export
use_point = transmitter_on & ~in_turn if remove_turns else transmitter_on

# Conversion from GPS global latitude and longitude to UTM latitude and longitude (in [m])
utm_lat, utm_lon, zone_num, zone_letter = utm.from_latlon(gps_lat, gps_lon)

for k, s in enumerate(scans):
    print(f"Scan {k} ({s.name}): {len(s.gps_times)} GPS points, {s.in_turn.sum()} in turns, "
          f"{(~s.transmitter_on).sum()} with transmitter off")

Mapping of Secondary Coil Voltages and Complex Field Phase

In [ ]:
import contextily as ctx
from matplotlib.ticker import FuncFormatter

# Satellite basemap source: Google satellite tiles, or Esri World Imagery as a fallback
basemap_source = 'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}'
# basemap_source = ctx.providers.Esri.WorldImagery
basemap_zoom = 20  # tile zoom level; 20 is the sharpest Google offers, lower it for large surveys
point_alpha = 0.7  # opacity of the data points over the satellite image
map_padding = 15   # extra map shown around the survey, in [m]

# utm_lat/utm_lon hold UTM easting/northing in [m]; tell the basemap which UTM zone they are in
utm_epsg = f"EPSG:{(32600 if gps_lat[0] >= 0 else 32700) + zone_num}"
utm_origin_x, utm_origin_y = np.min(utm_lat), np.min(utm_lon)

def plot_satellite_map(x, y, values, title, colorbar_label):
    fig, ax = plt.subplots(figsize=(9, 8))
    mag_map = ax.scatter(x, y, c=values, cmap='viridis', s=25, alpha=point_alpha,
                         edgecolors='white', linewidths=0.3, zorder=2)
    ax.set_aspect('equal')
    ax.set_xlim(np.min(x) - map_padding, np.max(x) + map_padding)
    ax.set_ylim(np.min(y) - map_padding, np.max(y) + map_padding)
    ctx.add_basemap(ax, crs=utm_epsg, source=basemap_source, zoom=basemap_zoom, attribution_size=6, zorder=1)
    # Label axes as metres from the survey's south-west corner instead of full UTM values
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v - utm_origin_x:.0f}"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v - utm_origin_y:.0f}"))
    fig.colorbar(mag_map, ax=ax, label=colorbar_label, shrink=0.8)
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    ax.set_title(title)
    plt.show()

# Envelope maps also leave out the first and last 10 GPS points of each scan
envelope_mask = use_point & (position_in_scan >= 10) & (position_from_scan_end >= 10)
plot_satellite_map(utm_lat[envelope_mask], utm_lon[envelope_mask], mag_ch1_fundamental_envelope_at_gps_coords[envelope_mask],
                   'Ch 1 Fundamental Envelope Map', 'Voltage (V)')
plot_satellite_map(utm_lat[envelope_mask], utm_lon[envelope_mask], mag_ch2_fundamental_envelope_at_gps_coords[envelope_mask],
                   'Ch 2 Fundamental Envelope Map', 'Voltage (V)')

# Turns (when remove_turns is on) and points recorded after the transmitter shut off are left out
rms_mask = use_point
plot_satellite_map(utm_lat[rms_mask], utm_lon[rms_mask], mag_ch1_voltages_at_gps_coords[rms_mask],
                   'Ch 1 RMS Voltage Map', 'Voltage (V)')
plot_satellite_map(utm_lat[rms_mask], utm_lon[rms_mask], mag_ch2_voltages_at_gps_coords[rms_mask],
                   'Ch 2 RMS Voltage Map', 'Ch 2 Voltage (V)')

mag_field_phase = np.arctan2(mag_ch2_voltages_at_gps_coords, mag_ch1_voltages_at_gps_coords)
plot_satellite_map(utm_lat[rms_mask], utm_lon[rms_mask], mag_field_phase[rms_mask], 'Phase Map', 'arctan(Ch2/Ch1)')

mag_map = plt.scatter(x=mag_ch1_voltages_at_gps_coords[rms_mask], y=mag_ch2_voltages_at_gps_coords[rms_mask], c=utm_lat[rms_mask]%1000, cmap='viridis')
plt.colorbar(mag_map, label='arctan(Ch2/Ch1)')
plt.xlabel('Ch1 Voltage (V)')
plt.ylabel('Ch2 Voltage (V)')
plt.title('Ch1 vs Ch2 RMS Voltage Map')
plt.axis('equal')
plt.grid(True)
plt.show()

In [ ]:
logname = "arcgis_data.csv"
path = os.path.expanduser('data/'+logname)
logfile = open(path, "w")
logfile.write("Times (s),GPS Lat,GPS Lon,Ch 1 Voltage (V),Ch 2 Voltage (V),Phase,Scan\n")
# Exports the same points that are on the RMS and phase maps
for i in np.flatnonzero(rms_mask):
        logfile.write(f"{gps_times[i]},{gps_lat[i]},{gps_lon[i]},\
                        {mag_ch1_voltages_at_gps_coords[i]:.7f},{mag_ch2_voltages_at_gps_coords[i]:.7f},{mag_field_phase[i]:.7f},{scan_index[i]}\n")
logfile.close()